# Lighter Live Stream

Stream live **Lighter.xyz** order books and trades from the [0xArchive WebSocket API](https://www.0xarchive.io/docs/websocket), capture a short window, and summarize it.

This notebook opens one connection to `wss://api.0xarchive.io/ws`, subscribes to two live Lighter channels, and creates:
- **Top-of-book view** with best bid, best ask, and mid from full top-20 books
- **Spread and depth** as quoted spread in basis points and top-20 size on each side
- **Trade flow** with taker buy and sell volume, counting each trade once by `tid`
- **Capture summary** of messages received, distinct trades, and volume

It sends the WebSocket commands directly with the `websockets` package, so every command and message shape is visible in the code.

**Requirements:** API key from [0xarchive.io/dashboard](https://0xarchive.io/dashboard). Live Lighter channels are available on every tier, including Free. Each delivered message is metered like other WebSocket data, so a longer capture or a shorter book interval uses more credits. See [Credits](https://www.0xarchive.io/docs/core-concepts/credits).

## 1. Setup

In [ ]:
%pip install websockets pandas matplotlib seaborn python-dotenv -q

In [ ]:
import asyncio
import json
import os
import time

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from websockets.asyncio.client import connect

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# --- Configuration ---
# Load API key from .env file (copy .env.example to .env and add your key)
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=True)

API_KEY = os.environ.get("OXARCHIVE_API_KEY", "your_api_key_here")
if API_KEY == "your_api_key_here":
    raise ValueError("Set OXARCHIVE_API_KEY in .env or as an environment variable")

# Lighter channels are served on api.0xarchive.io (not on stream.0xarchive.io)
WS_URL = "wss://api.0xarchive.io/ws"
SYMBOL = "BTC"            # Any symbol from GET /v1/lighter/instruments
CAPTURE_SECONDS = 60      # How long to stream before summarizing
BOOK_INTERVAL_MS = 500    # At most one book per interval: 100 to 5000 ms (1000 if omitted)
FLOW_BUCKET = "5s"        # Bucket size for the trade-flow chart

# Color palette
COLOR_BID = "#2ecc71"     # green  - best bid, taker buys
COLOR_ASK = "#e74c3c"     # red    - best ask, taker sells
COLOR_MID = "#f39c12"     # amber  - mid price
COLOR_SPREAD = "#9b59b6"  # purple - spread

print(f"Streaming Lighter {SYMBOL} for {CAPTURE_SECONDS}s from {WS_URL}")
print(f"Order book: at most one top-20 book every {BOOK_INTERVAL_MS} ms")

## 2. Subscribe and Capture

Live Lighter channels use the same commands and envelope as live Hyperliquid channels. The API key goes in the opening handshake as `Authorization: Bearer <key>`, then the client sends one `subscribe` command per channel:

```json
{"op": "subscribe", "channel": "lighter_orderbook", "symbol": "BTC", "interval_ms": 500}
{"op": "subscribe", "channel": "lighter_trades", "symbol": "BTC"}
```

The server confirms each one with `{"type": "subscribed", ...}` and then sends `{"type": "data", "channel": ..., "symbol": ..., "data": ...}` messages:

- **`lighter_orderbook`**: `data` is a full top-20 book, not a diff: `{"coin", "time", "levels": [bids, asks]}`. Bids are best (highest) first and asks best (lowest) first, up to 20 levels per side. Each level is `{"px", "sz", "n"}`: `px` and `sz` are decimal strings, and `n` is always 1 because Lighter does not publish per-level order counts. `interval_ms` is optional (100 to 5000); without it the server sends at most one book per second.
- **`lighter_trades`**: `data` is an array of fills. Every trade appears as **two legs**, one per side, sharing the same `tid`. `crossed` is `true` on the taker leg, and `side` is `B` for the bid side and `A` for the ask side.

Rejected commands and stream notices arrive as `{"type": "error", "message": ...}`. The capture below prints them.

In [ ]:
SUBSCRIPTIONS = [
    {"op": "subscribe", "channel": "lighter_orderbook", "symbol": SYMBOL, "interval_ms": BOOK_INTERVAL_MS},
    {"op": "subscribe", "channel": "lighter_trades", "symbol": SYMBOL},
]


async def capture(seconds):
    """Stream live Lighter data for `seconds`; return books, trade legs, and server notices."""
    books, trade_legs, notices = [], [], []
    headers = {"Authorization": f"Bearer {API_KEY}"}

    async with connect(WS_URL, additional_headers=headers) as ws:
        for command in SUBSCRIPTIONS:
            await ws.send(json.dumps(command))

        deadline = time.monotonic() + seconds
        while (remaining := deadline - time.monotonic()) > 0:
            try:
                raw = await asyncio.wait_for(ws.recv(), timeout=remaining)
            except asyncio.TimeoutError:
                break
            msg = json.loads(raw)
            kind = msg.get("type")

            if kind == "subscribed":
                print(f"Subscribed: {msg['channel']} {msg['symbol']}")
            elif kind == "error":
                notices.append(msg.get("message", ""))
                print(f"Server notice: {msg.get('message')}")
            elif kind == "data" and msg.get("channel") == "lighter_orderbook":
                books.append(msg["data"])
            elif kind == "data" and msg.get("channel") == "lighter_trades":
                trade_legs.extend(msg["data"])
    # Leaving the `async with` block closes the connection, which ends both subscriptions.

    return books, trade_legs, notices


# Jupyter runs cells inside an event loop, so the coroutine is awaited directly.
# In a plain Python script, use: asyncio.run(capture(CAPTURE_SECONDS))
books, trade_legs, notices = await capture(CAPTURE_SECONDS)
print(f"Captured {len(books):,} books and {len(trade_legs):,} trade legs")

## 3. Build DataFrames

Each order-book message replaces the previous one, so every book becomes one row with its best bid, best ask, mid, spread, and top-20 size per side.

For trades, the two legs of a trade share a `tid`. We keep one row per `tid`, preferring the taker leg (`crossed=True`) so its `side` gives the taker direction: `B` means the taker bought and `A` means the taker sold. Volume is summed over one leg per trade; adding both legs would double it.

In [ ]:
if not books:
    raise RuntimeError(
        "No order books received. Check SYMBOL against GET /v1/lighter/instruments "
        "or raise CAPTURE_SECONDS for a quieter market."
    )


def book_row(book):
    bids, asks = book["levels"]
    return {
        "time": pd.to_datetime(book["time"], unit="ms", utc=True),
        "best_bid": float(bids[0]["px"]) if bids else float("nan"),
        "best_ask": float(asks[0]["px"]) if asks else float("nan"),
        "bid_depth": sum(float(level["sz"]) for level in bids),
        "ask_depth": sum(float(level["sz"]) for level in asks),
    }


books_df = pd.DataFrame([book_row(b) for b in books]).set_index("time").sort_index()
books_df["mid"] = (books_df["best_bid"] + books_df["best_ask"]) / 2
books_df["spread_bps"] = (books_df["best_ask"] - books_df["best_bid"]) / books_df["mid"] * 10_000

legs = pd.DataFrame(trade_legs, columns=["tid", "time", "side", "px", "sz", "crossed"])
legs["px"] = legs["px"].astype(float)
legs["sz"] = legs["sz"].astype(float)
legs["time"] = pd.to_datetime(legs["time"], unit="ms", utc=True)

# One row per trade: sort taker legs first, then keep the first leg of each tid
trades = (
    legs.sort_values("crossed", ascending=False)
    .drop_duplicates("tid")
    .sort_values("time")
    .set_index("time")
)
trades["taker"] = trades["side"].map({"B": "Buy", "A": "Sell"}).where(trades["crossed"].astype(bool), "Unknown")
trades["notional"] = trades["px"] * trades["sz"]

print(f"Books:         {len(books_df):,}")
print(f"Trade legs:    {len(legs):,}")
print(f"Trades (tid):  {len(trades):,}")
if not trades.empty:
    print(trades["taker"].value_counts().to_string())

## 4. Top of Book, Spread, and Depth

Best bid, best ask, and mid from each book, with the quoted spread in basis points and the total size resting in the top 20 levels on each side.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1, 1]})

# Top: best bid, best ask, and mid
axes[0].step(books_df.index, books_df["best_bid"], where="post", color=COLOR_BID,
             linewidth=1.2, label="Best bid")
axes[0].step(books_df.index, books_df["best_ask"], where="post", color=COLOR_ASK,
             linewidth=1.2, label="Best ask")
axes[0].plot(books_df.index, books_df["mid"], color=COLOR_MID, linewidth=0.8,
             alpha=0.7, label="Mid")
axes[0].set_ylabel("Price")
axes[0].set_title(f"Lighter {SYMBOL} Live Top of Book ({CAPTURE_SECONDS}s capture)", fontsize=16)
axes[0].legend(loc="upper left", fontsize=12)

# Middle: quoted spread
axes[1].step(books_df.index, books_df["spread_bps"], where="post", color=COLOR_SPREAD, linewidth=1.2)
axes[1].set_ylabel("Spread (bps)")

# Bottom: top-20 size per side
axes[2].step(books_df.index, books_df["bid_depth"], where="post", color=COLOR_BID,
             linewidth=1.2, label="Bid size (top 20)")
axes[2].step(books_df.index, books_df["ask_depth"], where="post", color=COLOR_ASK,
             linewidth=1.2, label="Ask size (top 20)")
axes[2].set_ylabel("Size (base units)")
axes[2].set_xlabel("Time (UTC)")
axes[2].legend(loc="upper left", fontsize=12)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))

plt.tight_layout()
plt.show()

## 5. Trade Flow

Each trade is plotted at its price against the live mid and sized by trade size, above taker buy and taker sell volume per bucket. Every trade is counted once by `tid`.

In [ ]:
if trades.empty:
    print("No trades in the capture window. Raise CAPTURE_SECONDS or pick a more active symbol.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True,
                             gridspec_kw={"height_ratios": [2, 1]})

    # Top: trade prints over the mid
    axes[0].plot(books_df.index, books_df["mid"], color=COLOR_MID, linewidth=1.0,
                 alpha=0.8, label="Mid")
    size_scale = 300 / trades["sz"].max()
    for taker, color in [("Buy", COLOR_BID), ("Sell", COLOR_ASK), ("Unknown", "gray")]:
        subset = trades[trades["taker"] == taker]
        if not subset.empty:
            axes[0].scatter(subset.index, subset["px"], s=10 + subset["sz"] * size_scale,
                            color=color, alpha=0.6, label=f"Taker {taker.lower()}")
    axes[0].set_ylabel("Price")
    axes[0].set_title(f"Lighter {SYMBOL} Live Trades", fontsize=16)
    axes[0].legend(loc="upper left", fontsize=12)

    # Bottom: taker buy vs sell volume per bucket
    buy_vol = trades.loc[trades["taker"] == "Buy", "sz"].resample(FLOW_BUCKET).sum()
    sell_vol = trades.loc[trades["taker"] == "Sell", "sz"].resample(FLOW_BUCKET).sum()
    width = pd.Timedelta(FLOW_BUCKET) * 0.8
    axes[1].bar(buy_vol.index, buy_vol.values, width=width, align="edge",
                color=COLOR_BID, alpha=0.7, label="Taker buy")
    axes[1].bar(sell_vol.index, -sell_vol.values, width=width, align="edge",
                color=COLOR_ASK, alpha=0.7, label="Taker sell")
    axes[1].axhline(0, color="white", linewidth=0.5, alpha=0.3)
    axes[1].set_ylabel(f"Volume per {FLOW_BUCKET} (base units)")
    axes[1].set_xlabel("Time (UTC)")
    axes[1].legend(loc="upper left", fontsize=12)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))

    plt.tight_layout()
    plt.show()

## 6. Summary Statistics

In [ ]:
volume = trades["sz"].sum()
notional = trades["notional"].sum()
buy_volume = trades.loc[trades["taker"] == "Buy", "sz"].sum()
sell_volume = trades.loc[trades["taker"] == "Sell", "sz"].sum()

print(f"{'='*70}")
print(f"  LIGHTER {SYMBOL} LIVE CAPTURE: {CAPTURE_SECONDS}s")
print(f"  {books_df.index.min():%Y-%m-%d %H:%M:%S} to {books_df.index.max():%H:%M:%S} UTC (book times)")
print(f"{'='*70}")

print(f"\n  Order Book (interval {BOOK_INTERVAL_MS} ms)")
print(f"  {'-'*40}")
print(f"  Books received:         {len(books_df):>12,}")
print(f"  Last mid:               {books_df['mid'].iloc[-1]:>12,.2f}")
print(f"  Mean spread (bps):      {books_df['spread_bps'].mean():>12.3f}")
print(f"  Max spread (bps):       {books_df['spread_bps'].max():>12.3f}")

print("\n  Trades")
print(f"  {'-'*40}")
print(f"  Trade legs received:    {len(legs):>12,}")
print(f"  Distinct trades (tid):  {len(trades):>12,}")
print(f"  Volume (base units):    {volume:>12,.5f}")
print(f"  Notional:               {notional:>12,.2f}")
print(f"  Taker buy volume:       {buy_volume:>12,.5f}")
print(f"  Taker sell volume:      {sell_volume:>12,.5f}")
if notional > 0:
    print(f"  VWAP:                   {notional / volume:>12,.2f}")

print(f"\n  Server notices:         {len(notices):>12}")
for notice in notices:
    print(f"    - {notice}")

## Next Steps

- **More live channels:** `lighter_open_interest` and `lighter_funding` carry the same market-context message (open interest, funding rate, mark, index and mid prices, 24-hour volume), updated about once per second per market. `lighter_candles` and `lighter_l3_orderbook` are replay only; a live subscribe to them returns an error.
- **Live trades are preliminary:** `fee`, `fee_token`, `closed_pnl` and `dir` are always `null` in live messages. The finalized record, including fees, is served by `GET /v1/lighter/trades/{symbol}`, which returns only reconciled trades (see `meta.finalized_through`). `GET /v1/lighter/trades/{symbol}/recent` serves the preliminary tier.
- **Historical replay:** send `{"op": "replay", ...}` with the same channel names to replay stored history. Replay returns `historical_data` rows in the stored Lighter shape, which differs from the live shape used here, so parse the two separately.
- **Long-running clients:** keep a set of processed `tid` values so each trade is counted once across messages, and keep heavy work out of the receive loop. A client that falls behind `lighter_trades` gets an `error` notice saying how many messages were dropped; those trades are not resent, so fill the gap from the REST trade routes. A client that stays behind has that subscription stopped and must subscribe again.

See the [WebSocket docs](https://www.0xarchive.io/docs/websocket) for connection handling, limits, and replay controls.